# 🏋️ 헬스 기록 분석기 v0.1

아이폰 메모에 적던 운동 기록을 **그대로 붙여넣으면** → 표로 바꾸고 → 성장을 계산하고 → **다음 운동의 무게**를 제안해요.

**사용법**
1. 아래 ①의 `MEMO` 칸에 내 메모를 붙여넣기 (지금은 가짜 예시가 들어 있어요)
2. 메뉴 `런타임(Runtime) → 모두 실행(Run all)`
3. 맨 아래 🔧 에러 연습 셀에서 빨간 에러가 나며 멈추는 건 **정상**이에요 (일부러 만든 에러)

**메모 쓰는 법** — 아래 세 가지를 섞어 써도 돼요
```
9/28 가슴 삼두             ← 날짜 줄: '월/일'로 시작 (뒤에 쓰는 글자는 자유)
BenchPress 60*12*3         ← 무게*횟수*세트수 = 같은 세트 3번
벤치프레스 60*12,12,10     ← 무게*횟수 뒤에 횟수만 이어 쓰기 = 같은 무게로 한 세트씩
벤치프레스 60x12 60x12 62.5x10   ← 세트마다 무게*횟수
```
- `*` 대신 `x`, `×`도 돼요. 맨몸 운동은 `0*15`처럼 무게를 0으로.
- 종목 이름을 영어/한글/줄임말로 다르게 써도 ②의 `ALIASES`에 적어 두면 같은 종목으로 합쳐요.
- 세트 뒤에 쓴 글자(예: `오늘 쉬웠음`)는 무시해요. 단, **세트 뒤에 숫자만 따로 쓰면 횟수로 읽어요.**

**데이터 흐름**
```
메모(글자) → parse_memo() → rows(dict 목록) → df(표: 한 줄 = 한 세트) → ⑤요약 ⑥성장 ⑦주간 세트 ⑧다음 무게
```

## ① 메모 붙여넣기
`"""` 와 `"""` 사이에 메모를 붙여넣어요. 아래는 가짜 기록이에요.

`"""`(따옴표 3개)는 **여러 줄짜리 글자**의 시작과 끝 표시예요. 따옴표 1개(`"벤치"`)는 한 줄만 담을 수 있어서, 메모처럼 줄바꿈이 있는 글은 3개로 감싸요.

In [ ]:
MEMO = """
9/14 가슴 삼두
벤치프레스 60*8,8,8
인클라인 덤벨프레스 20x12 20x10 20x9
케이블 푸시다운 25*12,10,10

9/16 등 이두
랫풀다운 50*12,10,10
바벨로우 50*10*3
덤벨컬 10*12,10

9/18 하체 어깨
스쿼트 80*8,8,7
레그프레스 120*12*3
사이드 레터럴 레이즈 8*15,11

9/21 가슴 삼두
BenchPress 60*10,9,8
인클라인 덤벨프레스 20*12,11,10
케이블 푸시다운 25*12,12,10

9/23 등 이두
랫풀다운 50*12,12,11
바벨로우 50*12,10,10
시티드 로우 45*12*3
덤벨컬 10*12,11

9/25 하체 어깨
스쿼트 85*6,5,5,5
레그프레스 130*12,10,10
레그 컬 40*12*3
사이드 레터럴 레이즈 8*15,13
러닝머신 20분

9/28 가슴 삼두 (컨디션 좋음)
BenchPress 60*10*3
인클라인 덤벨프레스 20*12,12,11
케이블 푸시다운 25*12*3 오늘 쉬웠음
펙덱 플라이 40*12,12
"""

YEAR = 2026  # 메모에는 연도를 안 쓰니까 여기서 정해요

## ② 종목 설정: 부위 · 반복 범위 · 증량 무게
종목마다 한 줄씩. **여기만 고치면** 아래 결과가 전부 바뀌어요.
- **범위** `(최소, 최대)`: 모든 세트가 **최대**에 도달하면 증량. 예) `(8, 10)` 60kg → 8/8/8 … 10/10/10 달성 → 65kg
- **증량**: 한 번에 올릴 무게(kg). 2~5kg 사이로, 종목마다 다르게 정할 수 있어요.
- 설정에 없는 종목은 `DEFAULT` 값을 써요.

| 기준 | 값 | 근거 |
|---|---|---|
| 부위별 주간 세트 하한 | 10세트 | Schoenfeld 외 2017 메타분석: 주 10세트 이상이 그보다 적을 때보다 근비대가 컸다 |
| 부위별 주간 세트 상한 | 상한없음 | 현장에서 흔히 쓰는 값 (연구로 정해진 선은 아님) |
| 증량 규칙 | 목표달성 → 증량 | ACSM 2009 권고: 목표 횟수를 1-2회 넘기는 날이 2회 연속이면 2~10% 증량 |
| 목표 반복 범위 | 종목마다 (`EXERCISES`의 범위) | 관행적인 범위. 실패 근처까지 하면 넓은 범위(저중량~고중량)에서 근비대는 비슷 (Schoenfeld 외 2017) |

In [ ]:
import pandas as pd
from datetime import date, timedelta

# 쓸 수 있는 범위 예: (4, 6) (6, 8) (8, 10) (10, 12) (12, 15) (15, 20)
EXERCISES = {
    "벤치프레스":         {"부위": "가슴", "범위": (8, 10),  "증량": 5},
    "인클라인 덤벨프레스": {"부위": "가슴", "범위": (10, 12), "증량": 2},
    "케이블 푸시다운":     {"부위": "삼두", "범위": (10, 12), "증량": 2.5},
    "랫풀다운":           {"부위": "등",   "범위": (10, 12), "증량": 5},
    "바벨로우":           {"부위": "등",   "범위": (8, 10),  "증량": 5},
    "시티드 로우":         {"부위": "등",   "범위": (10, 12), "증량": 5},
    "덤벨컬":             {"부위": "이두", "범위": (10, 12), "증량": 2},
    "스쿼트":             {"부위": "하체", "범위": (6, 8),   "증량": 5},
    "레그프레스":         {"부위": "하체", "범위": (10, 12), "증량": 5},
    "레그 컬":            {"부위": "하체", "범위": (10, 12), "증량": 2.5},
    "사이드 레터럴 레이즈": {"부위": "어깨", "범위": (15, 20), "증량": 2},
}

# 같은 종목의 다른 이름 → EXERCISES에 있는 이름
ALIASES = {
    "BenchPress": "벤치프레스",
    "벤치": "벤치프레스",
}

DEFAULT = {"부위": "미분류", "범위": (8, 12), "증량": 2.5}   # 설정에 없는 종목

## ③ 메모 → 세트 목록 (`parse_memo`)
메모를 한 줄씩 읽으면서 셋 중 하나로 판단해요: **빈 줄** / **날짜 줄** / **운동 줄**.
- 날짜 줄을 만나면 "지금 날짜"를 바꿔 두고, 그 아래 운동 줄들은 모두 그 날짜로 기록해요.
- `60*12*3`은 **세트 3개 = 표의 3줄**, `60*12,12,10`도 **3줄**이 돼요.
- 이해 못 한 줄은 멈추지 않고 건너뛰고, 목록으로 알려 줘요.

In [ ]:
def parse_date(token):
    """'9/14' → date(2026, 9, 14). 날짜 모양이 아니면 None."""
    parts = token.split("/")
    if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
        return date(YEAR, int(parts[0]), int(parts[1]))
    return None


def parse_set(token):
    """'60x10' → (60.0, 10, 1),  '60*10*3' → (60.0, 10, 3).  세트 모양이 아니면 None."""
    parts = token.lower().replace("*", "x").replace("×", "x").split("x")
    if len(parts) not in (2, 3):
        return None
    if not parts[0].replace(".", "", 1).isdigit():     # 무게: 60, 62.5 처럼 소수점 1개까지
        return None
    if not all(p.isdigit() for p in parts[1:]):        # 횟수·세트수: 정수만
        return None
    weight = float(parts[0])
    reps = int(parts[1])
    count = int(parts[2]) if len(parts) == 3 else 1
    return weight, reps, count


def parse_memo(memo):
    rows = []             # 결과: 세트 하나 = dict 하나
    skipped = []          # 이해 못 한 줄
    current_date = None   # 지금 읽고 있는 날짜

    for line in memo.splitlines():
        tokens = line.replace(",", " ").split()
        if not tokens:                        # 빈 줄
            continue

        new_date = parse_date(tokens[0])
        if new_date:                          # 날짜 줄
            current_date = new_date
            continue

        name_words, sets = [], []             # 운동 줄
        for token in tokens:
            s = parse_set(token)
            if s:
                sets.append(s)
            elif token.isdigit() and sets:    # 60*12,12,10 의 '12', '10' → 앞 세트와 같은 무게
                sets.append((sets[-1][0], int(token), 1))
            elif not sets:                    # 첫 세트가 나오기 전 글자 = 종목 이름
                name_words.append(token)

        if current_date is None or not sets or not name_words:
            skipped.append(line)
            continue

        name = " ".join(name_words)
        name = ALIASES.get(name, name)        # 별명이면 정식 이름으로

        set_no = 0
        for weight, reps, count in sets:
            for _ in range(count):            # 60*10*3 → 3번 반복해서 3줄
                set_no += 1
                rows.append({"날짜": current_date, "종목": name,
                             "세트": set_no, "무게": weight, "횟수": reps})
    return rows, skipped


rows, skipped = parse_memo(MEMO)
print(f"세트 {len(rows)}개를 읽었어요. 첫 번째:", rows[0])
for line in skipped:
    print("⚠️ 이해 못 한 줄 (건너뜀):", line.strip())

## ④ 표 만들기 + 계산 열 추가
`rows`(dict 목록)를 pandas 표(`df`)로 바꾸고 열 3개를 더해요.
- **부위**: ②의 `EXERCISES`에서 찾기. 없으면 멈추지 않고 `"미분류"`
- **볼륨** = 무게 × 횟수
- **e1RM**(추정 1회 최대 무게) = 무게 × (1 + 횟수/30) — Epley 공식. *무게와 횟수가 다른 세트끼리 비교하려고* 써요. (횟수가 많을수록 부정확해져요)

In [ ]:
def setting_of(name):
    """종목 설정을 돌려준다. 설정에 없으면 DEFAULT."""
    return EXERCISES.get(name, DEFAULT)


df = pd.DataFrame(rows)
df["부위"] = [setting_of(name)["부위"] for name in df["종목"]]
df["볼륨"] = df["무게"] * df["횟수"]
df["e1RM"] = (df["무게"] * (1 + df["횟수"] / 30)).round(1)
df = df[["날짜", "부위", "종목", "세트", "무게", "횟수", "볼륨", "e1RM"]]   # 열 순서 정리

for name in sorted(df.loc[df["부위"] == "미분류", "종목"].unique()):
    print(f'⚠️ 설정에 없는 종목: "{name}" → ②의 EXERCISES에 추가하세요 (지금은 기본값 사용)')

df.tail(10)

## ⑤ 날짜별 요약: 종목 수 · 총 세트 · 총 볼륨
`groupby("날짜")` = "같은 날짜끼리 묶어서" 계산해요. 한 줄이 한 세트라서 **줄 수 = 세트 수**예요.

In [ ]:
daily = df.groupby("날짜").agg(
    종목수=("종목", "nunique"),
    총세트=("세트", "count"),
    총볼륨=("볼륨", "sum"),
)
daily

## ⑥ 종목별 · 부위별 성장 (내 성취도)
같은 종목의 **첫 기록 날**과 **최근 기록 날**을 비교해요.
- **e1RM 변화**: 최고 힘이 늘었나?
- **볼륨 변화**: 같은 무게로 더 많이 했나? (무게는 그대로고 횟수만 늘어도 여기에 보여요)

In [ ]:
# 1단계: 종목·날짜별로 '그날 최고 e1RM'과 '그날 볼륨 합계'
per_day = df.groupby(["부위", "종목", "날짜"]).agg(
    e1RM=("e1RM", "max"),
    볼륨=("볼륨", "sum"),
).reset_index()

# 2단계: 종목별로 첫날 vs 최근 날 (날짜순으로 정렬돼 있어서 first = 첫날, last = 최근)
growth = per_day.groupby(["부위", "종목"]).agg(
    기록일수=("날짜", "count"),
    첫_e1RM=("e1RM", "first"),
    최근_e1RM=("e1RM", "last"),
    첫_볼륨=("볼륨", "first"),
    최근_볼륨=("볼륨", "last"),
)
growth["e1RM 변화(%)"] = ((growth["최근_e1RM"] / growth["첫_e1RM"] - 1) * 100).round(1)
growth["볼륨 변화(%)"] = ((growth["최근_볼륨"] / growth["첫_볼륨"] - 1) * 100).round(1)
display(growth)

print("부위별 평균 변화(%)")
display(growth.groupby("부위")[["e1RM 변화(%)", "볼륨 변화(%)"]].mean().round(1))

## ⑦ 부위별 주간 세트
**최근 7일**(기록의 마지막 날 포함) 동안 부위별로 몇 세트 했는지 세요. (판정 없이 숫자만)

In [ ]:
last_day = df["날짜"].max()
recent = df[df["날짜"] > last_day - timedelta(days=7)]      # 마지막 날 포함 7일

print(f"기간: {last_day - timedelta(days=6)} ~ {last_day}")
recent.groupby("부위").size().to_frame("주간 세트")          # 줄 수 = 세트 수

## ⑧ 다음 무게
종목마다 **가장 최근 기록의 본 세트**(그날 가장 무거운 무게로 한 세트)를 보고 정해요. 범위·증량은 ②의 설정을 따라요.

| 조건 | 다음 무게 |
|---|---|
| 모든 세트가 범위 **최대** 이상 (예: 8~10회 → 10/10/10) | + 증량 무게 |
| 가장 많이 한 세트도 범위 **최소** 미만 | − 증량 무게 |
| 그 외 | 유지, 최대 횟수까지 채우기 |
| 맨몸 운동 (무게 0) | 무게 대신 세트마다 1회씩 더 |

In [ ]:
def next_weight(weight, reps, low, high, step):
    """최근 본 세트(무게, [횟수들])와 설정(범위, 증량)으로 다음 무게와 이유를 돌려준다."""
    if weight == 0:                                   # 맨몸 운동은 무게 대신 횟수를 늘린다
        return 0, "맨몸 → 같은 세트 수로 세트마다 1회씩 더"
    if min(reps) >= high:
        return weight + step, f"모든 세트 {high}회 달성 → +{step:g}kg"
    if max(reps) < low:
        return max(weight - step, 0), f"최고 {max(reps)}회 < {low}회 → -{step:g}kg"
    return weight, f"같은 무게로 모든 세트 {high}회까지"


plans = []
for exercise, g in df.groupby("종목"):
    last = g[g["날짜"] == g["날짜"].max()]                   # 이 종목의 가장 최근 날
    top = last["무게"].max()                                 # 그날 가장 무거운 무게 = 본 세트
    reps = last.loc[last["무게"] == top, "횟수"].tolist()

    setting = setting_of(exercise)
    low, high = setting["범위"]
    new_weight, reason = next_weight(top, reps, low, high, setting["증량"])
    plans.append({
        "부위": setting["부위"],
        "종목": exercise,
        "범위": f"{low}~{high}회",
        "지난번": f"{top:g}kg × " + "/".join(str(r) for r in reps),
        "다음 무게": f"{new_weight:g}kg",
        "이유": reason,
    })

pd.DataFrame(plans).sort_values(["부위", "종목"], ignore_index=True)

## ⑨ (선택) 표를 CSV 파일로 저장
나중에 모바일 앱을 만들 때 이 표 모양(열 이름)이 그대로 설계도가 돼요.

In [ ]:
df.to_csv("gym_log.csv", index=False, encoding="utf-8-sig")   # utf-8-sig: 엑셀로 열어도 한글이 안 깨져요
print("저장했어요: 왼쪽 📁(파일) 탭 → gym_log.csv → ⋮ → 다운로드")
print("※ Colab 런타임이 끝나면 이 파일은 사라져요. 원본 기록은 아이폰 메모에 그대로 있어요.")

## 🔧 에러 연습 (일부러 에러를 내는 셀)
실행하기 **전에** 무슨 에러가 날지 한 줄로 예상해 보세요.

에러 메시지(Traceback) 읽는 순서:
1. **맨 마지막 줄**: `에러 종류: 설명` (예: `ValueError: ...`)
2. 위로 올라가며 **`---->` 화살표가 가리키는 내 코드 줄** 찾기
3. AI에게 전달: *"`parse_date` 함수의 `return date(...)` 줄에서 `ValueError: day is out of range for month`가 났어. 입력은 `9/31`이었어."*

In [ ]:
# 연습 1: 9월 31일은 없는 날짜예요. 메모에 오타로 9/31을 쓰면 이 에러가 나요.
parse_date("9/31")

In [ ]:
# 연습 2: 사전에 없는 이름을 [ ]로 찾으면? → setting_of()에서 EXERCISES[...] 대신 .get()을 쓴 이유
EXERCISES["펙덱 플라이"]

## ✅ 점검 질문
1. 메모의 `케이블 푸시다운 25*12,10,10` 한 줄은 표에서 몇 줄이 될까? ③의 어느 줄 때문일까?
2. `벤치프레스 60kg 10 10 8`이라고 쓰면 왜 "이해 못 한 줄"이 될까?
3. 경고에 나온 `펙덱 플라이`를 가슴 운동, 범위 12~15회, 증량 2.5kg로 설정하려면 어디에 무엇을 쓸까? → **직접 추가하고** 다시 실행해서 ⑦의 가슴 세트 수와 ⑧의 결과가 바뀌는지 확인
4. 벤치프레스를 6~8회, 증량 2.5kg으로 바꾸면 ⑧의 벤치프레스 결과는 어떻게 될까? (먼저 예상 → 고쳐서 실행 → 비교)